# Lab Sheet 11: Real-Time Mobile Camera Integration via Python & OpenCV
**Course:** Computer Vision Lab

**Objective:** Stream live video from a mobile phone camera to a PC, process the feed with Python + OpenCV, and apply real-time computer vision techniques.

### Tasks
1. Live Motion & Security Alarm Detector
2. Mobile QR Code & Barcode Scanner
3. Hand Gesture Desktop Controller
4. Smart Attendance System (Face Recognition)
5. Document Scanner with Perspective Transform
6. Portable Color & Defect Inspector
7. Virtual Whiteboard / Air Painting
8. Real-Time OCR Text Reader
9. Driver Drowsiness / Blink Monitor
10. Real-Time AR Filter Overlay

> **Usage:** Each task is self-contained. Run the setup cells, then run only the task you want. Press **`q`** in the video window to stop. Run these in a local Jupyter (not Colab), since `cv2.imshow` needs a desktop display.

## 0. Setup

### Phone streaming
- **Android:** install *IP Webcam*, tap **Start server**, note the URL (e.g. `http://192.168.1.5:8080`). Stream URL = `http://<ip>:8080/video`.
- **iOS / Android:** *iVCam* or *EpocCam* show the phone as a virtual webcam on the PC, so use an index like `0` or `1`.
- Phone and PC must be on the **same Wi-Fi network**.

### Install dependencies (run once)

In [ ]:
# Uncomment and run once
# !pip install opencv-python numpy mediapipe pyautogui pyzbar face_recognition pytesseract pygame
# Tesseract engine (needed for Task 8): https://github.com/UB-Mannheim/tesseract/wiki (Windows) | sudo apt install tesseract-ocr (Linux) | brew install tesseract (macOS)

In [ ]:
import cv2, os, csv, time, math
import numpy as np
from datetime import datetime

# ---- CHANGE THIS to your phone stream ----
STREAM_URL = "http://192.168.1.5:8080/video"   # IP Webcam
# STREAM_URL = 0                               # iVCam/EpocCam virtual webcam or laptop webcam

def open_stream(url=STREAM_URL):
    cap = cv2.VideoCapture(url)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)         # keep latency low
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open stream: {url}. Check IP/port and Wi-Fi.")
    return cap

def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def quit_pressed(delay=1):
    return (cv2.waitKey(delay) & 0xFF) == ord("q")

# Quick connection test: shows 1 frame then closes
cap = open_stream()
ok, frame = cap.read()
print("Stream OK, frame shape:", frame.shape if ok else None)
cap.release()

## 1. Live Motion & Security Alarm Detector
**Idea:** Compare each frame to the previous one with `cv2.absdiff()`, threshold, find contours, and ignore those smaller than `MIN_AREA`. If big motion is found, show an alert and save a snapshot.

In [ ]:
def motion_detector(min_area=1500, diff_thresh=25, cooldown=3, out_dir="snapshots"):
    os.makedirs(out_dir, exist_ok=True)
    cap = open_stream()
    prev, last_saved = None, 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (21, 21), 0)
        if prev is None:
            prev = gray; continue
        diff = cv2.absdiff(prev, gray)
        thresh = cv2.threshold(diff, diff_thresh, 255, cv2.THRESH_BINARY)[1]
        thresh = cv2.dilate(thresh, None, iterations=2)
        cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        motion = False
        for c in cnts:
            if cv2.contourArea(c) < min_area: continue
            motion = True
            x, y, w, h = cv2.boundingRect(c)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        if motion:
            cv2.putText(frame, "!! MOTION DETECTED !!", (20, 40),
                        cv2.FONT_HERSHEY_DUPLEX, 1, (0, 0, 255), 3)
            if time.time() - last_saved > cooldown:
                path = os.path.join(out_dir, f"alert_{datetime.now():%Y%m%d_%H%M%S}.jpg")
                cv2.imwrite(path, frame); last_saved = time.time()
                print("Snapshot saved:", path)
        prev = gray
        cv2.imshow("1 - Motion Detector (q to quit)", frame)
        if quit_pressed(): break
    cap.release(); cv2.destroyAllWindows()

motion_detector()

## 2. Mobile QR Code & Barcode Scanner for Inventory
**Idea:** Decode QR codes with `cv2.QRCodeDetector` (and barcodes with `pyzbar` if installed). Each new item is logged to `inventory_log.csv` with a timestamp; the same code is not re-logged within `dedupe_sec`.

In [ ]:
try:
    from pyzbar import pyzbar
    HAVE_PYZBAR = True
except Exception:
    HAVE_PYZBAR = False
print("pyzbar available (barcodes):", HAVE_PYZBAR)

def scanner(csv_path="inventory_log.csv", dedupe_sec=5):
    if not os.path.exists(csv_path):
        with open(csv_path, "w", newline="") as f:
            csv.writer(f).writerow(["timestamp", "type", "data"])
    qr = cv2.QRCodeDetector()
    cap = open_stream()
    seen = {}                                   # data -> last logged time
    while True:
        ok, frame = cap.read()
        if not ok: break
        found = []                              # (type, data, polygon points)
        data, pts, _ = qr.detectAndDecode(frame)
        if data and pts is not None:
            found.append(("QR", data, pts.reshape(-1, 2).astype(int)))
        if HAVE_PYZBAR:
            for b in pyzbar.decode(frame):
                poly = np.array([(p.x, p.y) for p in b.polygon], dtype=int)
                found.append((b.type, b.data.decode("utf-8", "ignore"), poly))

        for kind, data, poly in found:
            cv2.polylines(frame, [poly], True, (0, 255, 0), 3)
            cv2.putText(frame, f"{kind}: {data}", (poly[0][0], max(poly[0][1] - 10, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            if time.time() - seen.get(data, 0) > dedupe_sec:
                seen[data] = time.time()
                with open(csv_path, "a", newline="") as f:
                    csv.writer(f).writerow([now_str(), kind, data])
                print("Logged:", kind, data)
        cv2.imshow("2 - QR/Barcode Scanner (q to quit)", frame)
        if quit_pressed(): break
    cap.release(); cv2.destroyAllWindows()

scanner()

## 3. Hand Gesture Desktop Controller
**Idea:** Use MediaPipe Hands to get 21 landmarks, count raised fingers, and map the count to a PC action with `pyautogui`.

| Fingers | Action |
|---|---|
| 1 | Scroll up |
| 2 | Scroll down |
| 3 | Volume up |
| 4 | Volume down |
| 5 | Next slide (Right arrow) |

Set `DRY_RUN = False` to actually trigger system actions.

In [ ]:
import mediapipe as mp
import pyautogui

DRY_RUN = True          # True = only print actions (safe for testing)
ACTIONS = {
    1: ("Scroll Up",   lambda: pyautogui.scroll(300)),
    2: ("Scroll Down", lambda: pyautogui.scroll(-300)),
    3: ("Volume Up",   lambda: pyautogui.press("volumeup")),
    4: ("Volume Down", lambda: pyautogui.press("volumedown")),
    5: ("Next Slide",  lambda: pyautogui.press("right")),
}

def count_fingers(lm, label):
    # lm: 21 landmarks (normalized). label: "Left"/"Right" as seen by MediaPipe (mirrored image)
    tips, pips = [8, 12, 16, 20], [6, 10, 14, 18]
    n = sum(1 for t, p in zip(tips, pips) if lm[t].y < lm[p].y)
    # thumb: compare x of tip vs IP joint depending on hand
    if label == "Right": n += int(lm[4].x < lm[3].x)
    else:                n += int(lm[4].x > lm[3].x)
    return n

def gesture_controller(cooldown=1.0):
    mp_hands, mp_draw = mp.solutions.hands, mp.solutions.drawing_utils
    cap, last = open_stream(), 0
    with mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7,
                        min_tracking_confidence=0.6) as hands:
        while True:
            ok, frame = cap.read()
            if not ok: break
            frame = cv2.flip(frame, 1)
            res = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if res.multi_hand_landmarks:
                hl = res.multi_hand_landmarks[0]
                label = res.multi_handedness[0].classification[0].label
                n = count_fingers(hl.landmark, label)
                mp_draw.draw_landmarks(frame, hl, mp_hands.HAND_CONNECTIONS)
                name = ACTIONS[n][0] if n in ACTIONS else "-"
                cv2.putText(frame, f"Fingers: {n}  Action: {name}", (10, 40),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
                if n in ACTIONS and time.time() - last > cooldown:
                    last = time.time()
                    print("Action:", name)
                    if not DRY_RUN: ACTIONS[n][1]()
            cv2.imshow("3 - Gesture Controller (q to quit)", frame)
            if quit_pressed(): break
    cap.release(); cv2.destroyAllWindows()

gesture_controller()

## 4. Smart Attendance System via Face Recognition
**Idea:** Put one photo per person in `known_faces/` (file name = person name, e.g. `Aman.jpg`). The code encodes them, matches faces from the stream, and writes `attendance.csv` (once per person per day).

In [ ]:
import face_recognition

def load_known(dir_path="known_faces"):
    os.makedirs(dir_path, exist_ok=True)
    encs, names = [], []
    for fn in os.listdir(dir_path):
        if fn.lower().endswith((".jpg", ".jpeg", ".png")):
            img = face_recognition.load_image_file(os.path.join(dir_path, fn))
            e = face_recognition.face_encodings(img)
            if e:
                encs.append(e[0]); names.append(os.path.splitext(fn)[0])
    print("Loaded faces:", names)
    return encs, names

def attendance_system(csv_path="attendance.csv", tolerance=0.5):
    known_encs, known_names = load_known()
    if not known_encs:
        print("Add photos to known_faces/ first."); return
    marked = set()
    if os.path.exists(csv_path):
        today = datetime.now().strftime("%Y-%m-%d")
        with open(csv_path) as f:
            for row in csv.reader(f):
                if len(row) >= 2 and row[1].startswith(today): marked.add(row[0])
    else:
        with open(csv_path, "w", newline="") as f:
            csv.writer(f).writerow(["name", "timestamp"])

    cap = open_stream(); frame_i = 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        frame_i += 1
        if frame_i % 3 == 0:                               # process every 3rd frame for speed
            small = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
            rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
            locs = face_recognition.face_locations(rgb)
            encs = face_recognition.face_encodings(rgb, locs)
            for (t, r, b, l), enc in zip(locs, encs):
                d = face_recognition.face_distance(known_encs, enc)
                i = int(np.argmin(d))
                name = known_names[i] if d[i] <= tolerance else "Unknown"
                t, r, b, l = [v * 4 for v in (t, r, b, l)]
                color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
                cv2.rectangle(frame, (l, t), (r, b), color, 2)
                cv2.putText(frame, name, (l, t - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
                if name != "Unknown" and name not in marked:
                    marked.add(name)
                    with open(csv_path, "a", newline="") as f:
                        csv.writer(f).writerow([name, now_str()])
                    print("Attendance marked:", name)
        cv2.imshow("4 - Attendance (q to quit)", frame)
        if quit_pressed(): break
    cap.release(); cv2.destroyAllWindows()

attendance_system()

## 5. Document Scanner with Perspective Transform
**Idea:** Canny edges → largest 4-point contour → `getPerspectiveTransform` → flatten → adaptive threshold for a B/W scan.
Press **`s`** to scan the current frame (saved as `scan_*.png`), **`q`** to quit.

In [ ]:
def order_points(pts):
    pts = pts.reshape(4, 2).astype("float32")
    s, d = pts.sum(axis=1), np.diff(pts, axis=1).ravel()
    return np.array([pts[np.argmin(s)], pts[np.argmin(d)],
                     pts[np.argmax(s)], pts[np.argmax(d)]], dtype="float32")  # TL, TR, BR, BL

def find_document(frame):
    gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    edges = cv2.dilate(cv2.Canny(gray, 75, 200), None, iterations=1)
    cnts, _ = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    for c in sorted(cnts, key=cv2.contourArea, reverse=True)[:5]:
        approx = cv2.approxPolyDP(c, 0.02 * cv2.arcLength(c, True), True)
        if len(approx) == 4 and cv2.contourArea(approx) > 0.1 * frame.shape[0] * frame.shape[1]:
            return approx
    return None

def warp_document(frame, quad):
    tl, tr, br, bl = order_points(quad)
    W = int(max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl)))
    H = int(max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl)))
    dst = np.array([[0, 0], [W - 1, 0], [W - 1, H - 1], [0, H - 1]], dtype="float32")
    M = cv2.getPerspectiveTransform(np.array([tl, tr, br, bl], dtype="float32"), dst)
    warped = cv2.warpPerspective(frame, M, (W, H))
    g = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
    return cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 21, 10)

def document_scanner():
    cap, count = open_stream(), 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        quad = find_document(frame)
        view = frame.copy()
        if quad is not None:
            cv2.drawContours(view, [quad], -1, (0, 255, 0), 3)
        cv2.putText(view, "Press 's' to scan, 'q' to quit", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
        cv2.imshow("5 - Document Scanner", view)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("s") and quad is not None:
            scan = warp_document(frame, quad)
            fn = f"scan_{datetime.now():%Y%m%d_%H%M%S}.png"
            cv2.imwrite(fn, scan); count += 1
            cv2.imshow("Scanned", scan); print("Saved", fn)
        elif key == ord("q"): break
    cap.release(); cv2.destroyAllWindows()

document_scanner()

## 6. Portable Color & Defect Inspector
**Idea:** Convert BGR → HSV and isolate color ranges with `cv2.inRange`. Example: classify fruit as **ripe (red/yellow)** vs **unripe (green)** by the area of each color mask. Tune the ranges for your lighting.

In [ ]:
COLOR_RANGES = {   # name: list of (lower, upper) HSV ranges; BGR drawing color
    "Red (ripe)":     ([((0, 120, 70), (10, 255, 255)), ((170, 120, 70), (180, 255, 255))], (0, 0, 255)),
    "Yellow (ripe)":  ([((20, 100, 100), (35, 255, 255))], (0, 255, 255)),
    "Green (unripe)": ([((36, 60, 60), (85, 255, 255))], (0, 255, 0)),
}
MIN_OBJ_AREA = 2000

def color_inspector():
    cap = open_stream()
    kernel = np.ones((5, 5), np.uint8)
    while True:
        ok, frame = cap.read()
        if not ok: break
        hsv = cv2.cvtColor(cv2.GaussianBlur(frame, (7, 7), 0), cv2.COLOR_BGR2HSV)
        combined = np.zeros(frame.shape[:2], np.uint8)
        for name, (ranges, bgr) in COLOR_RANGES.items():
            mask = np.zeros(frame.shape[:2], np.uint8)
            for lo, hi in ranges:
                mask |= cv2.inRange(hsv, np.array(lo), np.array(hi))
            mask = cv2.morphologyEx(cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel), cv2.MORPH_CLOSE, kernel)
            combined |= mask
            cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for c in cnts:
                if cv2.contourArea(c) < MIN_OBJ_AREA: continue
                x, y, w, h = cv2.boundingRect(c)
                cv2.rectangle(frame, (x, y), (x + w, y + h), bgr, 2)
                cv2.putText(frame, name, (x, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, bgr, 2)
        masked = cv2.bitwise_and(frame, frame, mask=combined)
        cv2.imshow("6 - Color Inspector (q to quit)", np.hstack([frame, masked]))
        if quit_pressed(): break
    cap.release(); cv2.destroyAllWindows()

color_inspector()

## 7. Virtual Whiteboard / Air Painting
**Idea:** Track a **blue** object (marker cap) by HSV masking, find its centroid, and draw lines between successive positions on a separate canvas buffer, merged onto the video. Keys: **`c`** clear, **`q`** quit.

In [ ]:
def air_painter(lower=(100, 150, 70), upper=(130, 255, 255), min_radius=8):
    cap = open_stream()
    canvas, prev_pt = None, None
    while True:
        ok, frame = cap.read()
        if not ok: break
        frame = cv2.flip(frame, 1)
        if canvas is None: canvas = np.zeros_like(frame)
        hsv = cv2.cvtColor(cv2.GaussianBlur(frame, (11, 11), 0), cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, np.array(lower), np.array(upper))
        mask = cv2.dilate(cv2.erode(mask, None, iterations=2), None, iterations=2)
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        pt = None
        if cnts:
            c = max(cnts, key=cv2.contourArea)
            (x, y), r = cv2.minEnclosingCircle(c)
            if r > min_radius:
                pt = (int(x), int(y))
                cv2.circle(frame, pt, int(r), (0, 255, 255), 2)
        if pt and prev_pt:
            cv2.line(canvas, prev_pt, pt, (0, 0, 255), 6)
        prev_pt = pt
        out = cv2.addWeighted(frame, 1, canvas, 1, 0)
        cv2.imshow("7 - Air Painter (c=clear, q=quit)", out)
        cv2.imshow("Canvas", canvas)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("c"): canvas[:] = 0
        elif key == ord("q"): break
    cap.release(); cv2.destroyAllWindows()

air_painter()

## 8. Real-Time OCR Text Reader
**Idea:** Pre-process (grayscale + threshold) and run Tesseract every few frames (OCR is slow), then overlay the recognized text on the video.

If Tesseract isn't on your PATH (Windows), set `pytesseract.pytesseract.tesseract_cmd` below.

In [ ]:
import pytesseract
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

def ocr_reader(every_n=20):
    cap, i, text = open_stream(), 0, ""
    while True:
        ok, frame = cap.read()
        if not ok: break
        i += 1
        if i % every_n == 0:
            g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            g = cv2.threshold(cv2.GaussianBlur(g, (3, 3), 0), 0, 255,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
            text = " ".join(pytesseract.image_to_string(g, config="--psm 6").split())
            if text: print("OCR:", text)
        # overlay on a dark banner
        cv2.rectangle(frame, (0, 0), (frame.shape[1], 40), (0, 0, 0), -1)
        cv2.putText(frame, text[:70] or "(no text)", (10, 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.imshow("8 - OCR Reader (q to quit)", frame)
        if quit_pressed(): break
    cap.release(); cv2.destroyAllWindows()

ocr_reader()

## 9. Mobile Driver Drowsiness / Blink Monitor
**Idea:** With MediaPipe Face Mesh compute the Eye Aspect Ratio

$$EAR = \frac{\|p_2-p_6\| + \|p_3-p_5\|}{2\,\|p_1-p_4\|}$$

If EAR stays below `EAR_THRESH` for longer than `CLOSED_SECS`, sound an alarm.

In [ ]:
import mediapipe as mp
import platform

# MediaPipe Face Mesh eye landmark indices in order p1..p6
LEFT_EYE  = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]

def ear(pts):
    p1, p2, p3, p4, p5, p6 = pts
    return (np.linalg.norm(p2 - p6) + np.linalg.norm(p3 - p5)) / (2.0 * np.linalg.norm(p1 - p4))

def beep():
    try:
        if platform.system() == "Windows":
            import winsound; winsound.Beep(1500, 600)
        else:
            import pygame
            if not pygame.mixer.get_init(): pygame.mixer.init(frequency=22050, size=-16, channels=1)
            t = np.linspace(0, 0.6, 13230, False)
            wave = (np.sin(2 * np.pi * 1500 * t) * 32767).astype(np.int16)
            pygame.sndarray.make_sound(wave).play()
    except Exception:
        print("\a ALARM!")

def drowsiness_monitor(EAR_THRESH=0.21, CLOSED_SECS=2.0):
    mesh = mp.solutions.face_mesh.FaceMesh(max_num_faces=1, refine_landmarks=True)
    cap, closed_since, blinks, was_closed = open_stream(), None, 0, False
    while True:
        ok, frame = cap.read()
        if not ok: break
        h, w = frame.shape[:2]
        res = mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if res.multi_face_landmarks:
            lm = res.multi_face_landmarks[0].landmark
            get = lambda idx: [np.array([lm[i].x * w, lm[i].y * h]) for i in idx]
            L, R = get(LEFT_EYE), get(RIGHT_EYE)
            e = (ear(L) + ear(R)) / 2
            for p in L + R: cv2.circle(frame, tuple(p.astype(int)), 2, (0, 255, 0), -1)
            if e < EAR_THRESH:
                if closed_since is None: closed_since = time.time()
                was_closed = True
                dur = time.time() - closed_since
                if dur >= CLOSED_SECS:
                    cv2.putText(frame, "DROWSINESS ALERT!", (30, 100),
                                cv2.FONT_HERSHEY_DUPLEX, 1.4, (0, 0, 255), 3)
                    beep()
            else:
                if was_closed and closed_since and time.time() - closed_since < CLOSED_SECS: blinks += 1
                closed_since, was_closed = None, False
            cv2.putText(frame, f"EAR: {e:.2f}  Blinks: {blinks}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
        cv2.imshow("9 - Drowsiness Monitor (q to quit)", frame)
        if quit_pressed(): break
    cap.release(); cv2.destroyAllWindows()

drowsiness_monitor()

## 10. Real-Time AR Filter Overlay
**Idea:** Locate the outer eye corners with Face Mesh, compute the eye distance and head tilt angle, then rotate/scale a **transparent RGBA** "glasses" image and alpha-blend it onto the face. The glasses image is generated in code, so no external file is needed (swap in your own PNG with `cv2.imread(path, cv2.IMREAD_UNCHANGED)`).

In [ ]:
import mediapipe as mp

def make_glasses(w=400, h=160):
    img = np.zeros((h, w, 4), np.uint8)
    lens_r, cy = 62, h // 2
    for cx in (100, w - 100):
        cv2.circle(img, (cx, cy), lens_r, (255, 180, 0, 90), -1)      # semi-transparent lens
        cv2.circle(img, (cx, cy), lens_r, (20, 20, 20, 255), 10)       # frame
    cv2.line(img, (100 + lens_r, cy), (w - 100 - lens_r, cy), (20, 20, 20, 255), 10)  # bridge
    return img

def overlay_rgba(bg, fg, cx, cy, angle=0.0):
    h, w = fg.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    fg = cv2.warpAffine(fg, M, (w, h), flags=cv2.INTER_LINEAR, borderValue=(0, 0, 0, 0))
    x1, y1 = int(cx - w / 2), int(cy - h / 2)
    bx1, by1 = max(x1, 0), max(y1, 0)
    bx2, by2 = min(x1 + w, bg.shape[1]), min(y1 + h, bg.shape[0])
    if bx1 >= bx2 or by1 >= by2: return bg
    roi = fg[by1 - y1:by2 - y1, bx1 - x1:bx2 - x1]
    alpha = roi[..., 3:4] / 255.0
    bg[by1:by2, bx1:bx2] = (alpha * roi[..., :3] + (1 - alpha) * bg[by1:by2, bx1:bx2]).astype(np.uint8)
    return bg

def ar_filter():
    glasses = make_glasses()
    mesh = mp.solutions.face_mesh.FaceMesh(max_num_faces=2, refine_landmarks=True)
    cap = open_stream()
    while True:
        ok, frame = cap.read()
        if not ok: break
        frame = cv2.flip(frame, 1)
        h, w = frame.shape[:2]
        res = mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        for face in (res.multi_face_landmarks or []):
            lm = face.landmark
            a = np.array([lm[33].x * w,  lm[33].y * h])     # eye outer corner (image left)
            b = np.array([lm[263].x * w, lm[263].y * h])    # eye outer corner (image right)
            dist = np.linalg.norm(b - a)
            angle = math.degrees(math.atan2(b[1] - a[1], b[0] - a[0]))
            gw = int(dist * 1.5)
            if gw < 20: continue
            g = cv2.resize(glasses, (gw, int(gw * glasses.shape[0] / glasses.shape[1])))
            cx, cy = (a + b) / 2
            frame = overlay_rgba(frame, g, cx, cy, -angle)  # negative: image y-axis points down
        cv2.imshow("10 - AR Filter (q to quit)", frame)
        if quit_pressed(): break
    cap.release(); cv2.destroyAllWindows()

ar_filter()

## Cleanup & Troubleshooting
- **Stream won't open:** verify the IP, port and `/video` path in the phone's browser first; PC and phone must share Wi-Fi.
- **Laggy video:** lower the phone's streaming resolution, or process every Nth frame.
- **Window stuck:** run the cell below.
- **Modules missing:** re-run the install cell in Setup.

In [ ]:
cv2.destroyAllWindows()